In [2]:
import requests
import urllib3
import numpy as np
from tensorflow import keras

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
print("Librerie importate!")

Librerie importate!


In [3]:
def scarica_testo_dante():
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
    print(f"Scaricamento da: {url}...")
    try:
        response = requests.get(url, verify=False)
        response.raise_for_status()
        response.encoding = 'utf-8'
        text = response.text
        print(f"Download completato! Lunghezza: {len(text)} caratteri")
        return text
    except Exception as e:
        print(f"Errore: {e}, provo link backup...")
        url_backup = "https://raw.githubusercontent.com/wpm/t-sne-text-vis/master/data/divina_commedia.txt"
        r2 = requests.get(url_backup, verify=False)
        r2.encoding = 'utf-8'
        return r2.text

testo = scarica_testo_dante()

Scaricamento da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...
Download completato! Lunghezza: 551846 caratteri


In [4]:
caratteri_unici = sorted(set(testo))
num_car = len(caratteri_unici)
char_to_idx = {c: i for i, c in enumerate(caratteri_unici)}
idx_to_char = {i: c for i, c in enumerate(caratteri_unici)}
print(f"Caratteri unici: {num_car}")

Caratteri unici: 69


In [5]:
seq_length = 80
X, y = [], []
for i in range(0, len(testo) - seq_length):
    sequenza = testo[i:i + seq_length]
    carattere_successivo = testo[i + seq_length]
    X.append([char_to_idx[c] for c in sequenza])
    y.append(char_to_idx[carattere_successivo])

X = np.reshape(X, (len(X), seq_length, 1))
X = X / num_car
y = keras.utils.to_categorical(y, num_classes=num_car)
print(f"Dataset creato! Shape X: {X.shape}")

Dataset creato! Shape X: (551766, 80, 1)


In [6]:
modello = keras.Sequential([
    keras.layers.LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    keras.layers.Dropout(0.2),
    keras.layers.LSTM(256),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(num_car, activation='softmax')
])
modello.compile(loss='categorical_crossentropy', optimizer='adam')
modello.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 80, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 80, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 69)             │        17,733 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 807,237 (3.08 MB)

 Trainable params: 807,237 (3.08 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
print("Inizio addestramento...")
storico = modello.fit(X, y, epochs=10, batch_size=128, verbose=1)
print("Addestramento completato!")

Inizio addestramento...
Epoch 1/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 118s 27ms/step - loss: 2.5854
Epoch 2/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 119s 28ms/step - loss: 2.3251
Epoch 3/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 119s 28ms/step - loss: 2.1798
Epoch 4/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 143s 28ms/step - loss: 2.0776
Epoch 5/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 119s 28ms/step - loss: 2.0047
Epoch 6/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 119s 28ms/step - loss: 1.9513
Epoch 7/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 142s 28ms/step - loss: 1.9057
Epoch 8/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 119s 28ms/step - loss: 1.8682
Epoch 9/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 120s 28ms/step - loss: 1.8359
Epoch 10/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 119s 28ms/step - loss: 1.8084
Addestramento completato!
